# 04 · Checkpoint 保存与恢复(★★★★★)

决赛容器易回收,checkpoint 保命。会:存完整字典、断点续训、只加载模型推理、config 匹配、去 `module.` 前缀、**存最优非最后一轮**。

In [ ]:
# ============ 公共设置(每个 notebook 先跑这一格)============
import os, sys, json, glob, math, time, numpy as np, torch, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")

BASE = "/public/home/xdzs2026_c296"          # ★你的主目录,若不同改这里
BASELINE = f"{BASE}/xiandao2026-AI4S/pangu_weather"   # 官方 baseline(含 maxvit3d_student.py, conf, data)
CKPT = f"{BASELINE}/data/checkpoints/model_bak.pth"   # 教师权重
TRAIN_DATA = f"{BASE}/era5_real"              # 训练数据(13年)
VAL_DATA   = f"{BASE}/era5_testc"             # 验证/测试数据(2000年)
WORK = f"{BASE}/_learn_work"                  # 本教程的工作目录(存中间文件)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, BASELINE)                  # 为了 import maxvit3d_student
print("torch", torch.__version__, "| DCU 可用:", torch.cuda.is_available())


In [ ]:
from maxvit3d_student import MaxVit3DStudent
dev = 0
cfg = {"embed":96,"depths":[2,4,2],"heads":[6,12,6],"patch":[2,16,16],"mlp_ratio":2.0}
student = MaxVit3DStudent(patch_size=tuple(cfg["patch"]), embed_dim=cfg["embed"], depths=tuple(cfg["depths"]),
                          num_heads=tuple(cfg["heads"]), mlp_ratio=cfg["mlp_ratio"]).to(dev)
opt = torch.optim.AdamW(student.parameters(), lr=6e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)

## 1) 保存完整 checkpoint(model + optimizer + scheduler + epoch + best + config)

In [ ]:
state = {"model_state_dict": student.state_dict(),
         "optimizer_state_dict": opt.state_dict(),
         "scheduler_state_dict": sched.state_dict(),
         "epoch": 7, "best_loss": 0.0966, "config": cfg}
torch.save(state, f"{WORK}/ckpt.pth")
print("已存", f"{WORK}/ckpt.pth")

## 2) 断点续训:恢复 model+optimizer+scheduler+epoch,接着训

In [ ]:
ck = torch.load(f"{WORK}/ckpt.pth", map_location=f"cuda:{dev}", weights_only=False)
student.load_state_dict(ck["model_state_dict"])
opt.load_state_dict(ck["optimizer_state_dict"])
sched.load_state_dict(ck["scheduler_state_dict"])
start_epoch = ck["epoch"]
print(f"从 epoch {start_epoch} 续训, best={ck['best_loss']}")

## 3) 只加载模型用于推理:按 config 重建 + 去 module. 前缀 + strict 检查

In [ ]:
ck = torch.load(f"{WORK}/ckpt.pth", map_location="cpu", weights_only=False)
c = ck["config"]                                    # ★用存下来的 config 重建,保证结构匹配
m = MaxVit3DStudent(patch_size=tuple(c["patch"]), embed_dim=c["embed"], depths=tuple(c["depths"]),
                    num_heads=tuple(c["heads"]), mlp_ratio=c["mlp_ratio"])
sd = ck["model_state_dict"]
sd = {k[7:] if k.startswith("module.") else k: v for k,v in sd.items()}   # DDP 会带 module. 前缀,去掉
missing, unexpected = m.load_state_dict(sd, strict=True)
print("strict 加载:", "OK" if not missing and not unexpected else f"缺{missing} 多{unexpected}")
assert not missing and not unexpected

### ✅ 要点:存**最优**(非最后一轮);推理按 config 重建;去 `module.` 前缀;strict=True 验证结构与权重匹配。